In [42]:
import pandas as pd
import numpy as np

FILE_PATH = "dataset/data_original_version.csv"

df = pd.read_csv(FILE_PATH)
print(f"Данные загружены. Размер: {df.shape}")
df.head()
df.info()

Данные загружены. Размер: (22676, 12)
<class 'pandas.DataFrame'>
RangeIndex: 22676 entries, 0 to 22675
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Price             22676 non-null  float64
 1   Apartment type    22676 non-null  str    
 2   Metro station     22676 non-null  str    
 3   Minutes to metro  22676 non-null  float64
 4   Region            22676 non-null  str    
 5   Number of rooms   22676 non-null  float64
 6   Area              22676 non-null  float64
 7   Living area       22676 non-null  float64
 8   Kitchen area      22676 non-null  float64
 9   Floor             22676 non-null  float64
 10  Number of floors  22676 non-null  int64  
 11  Renovation        22676 non-null  str    
dtypes: float64(7), int64(1), str(4)
memory usage: 2.1 MB


In [43]:
# 1.1 — описательная статистика по числовым признакам
df.describe()


,Price,Minutes to metro,Number of rooms,Area,Living area,Kitchen area,Floor,Number of floors
count,2.267600e+04,22676.000000,22676.000000,22676.000000,22676.000000,22676.000000,22676.000000,22676.000000
mean,3.612132e+07,11.888605,2.043129,71.966827,38.517953,12.594082,9.190466,16.556095
std,8.282561e+07,6.204457,1.523586,68.368608,38.124278,7.728074,7.549996,9.779297
min,1.150000e+06,0.000000,0.000000,6.000000,2.000000,1.000000,1.000000,1.000000
25%,7.068116e+06,7.000000,1.000000,37.400000,17.600000,8.600000,4.000000,11.000000
50%,1.134320e+07,11.000000,2.000000,53.300000,28.500000,10.600000,8.000000,16.000000
75%,2.479925e+07,15.000000,3.000000,77.140000,43.200000,14.500000,13.000000,20.000000
max,2.455020e+09,60.000000,12.000000,1117.000000,566.800000,122.000000,92.000000,97.000000


In [44]:
# 1.2 — что в категориальных колонках
for col in ['Apartment type', 'Region', 'Renovation', 'Metro station']:
    print(f"\n=== {col} ===")
    print(f"Уникальных значений: {df[col].nunique()}")
    print(df[col].value_counts().head(10))



=== Apartment type ===
Уникальных значений: 2
Apartment type
Secondary       13152
New building     9524
Name: count, dtype: int64

=== Region ===
Уникальных значений: 2
Region
Moscow           16113
Moscow region     6563
Name: count, dtype: int64

=== Renovation ===
Уникальных значений: 4
Renovation
Cosmetic                     12788
European-style renovation     3666
Without renovation            3198
Designer                      3024
Name: count, dtype: int64

=== Metro station ===
Уникальных значений: 547
Metro station
Красногвардейская    2697
Депо                 1646
Братиславская        1157
Котельники            981
Жулебино              731
Зябликово             716
Битца                 360
 Опалиха              305
Каширская             295
Домодедовская         272
Name: count, dtype: int64


In [45]:
# 1.3 — явные дубликаты строк
print(f"Полных дубликатов строк: {df.duplicated().sum()}")


Полных дубликатов строк: 1835


## Фильтрация и удаление дубликатов

In [46]:
# Сохраняем исходник
print(f"До фильтрации: {df.shape}")
# Удаление полных дубликатов
df = df.drop_duplicates().reset_index(drop=True)
print(f"После удаления дубликатов: {df.shape}")
# Оставляем только вторичку
df = df[df['Apartment type'] == 'Secondary'].reset_index(drop=True)
print(f"После фильтра 'Secondary': {df.shape}")
# Оставляем только Москву
df = df[df['Region'] == 'Moscow'].reset_index(drop=True)
print(f"После фильтра 'Moscow': {df.shape}")
df = df.drop(columns=['Apartment type', 'Region'])
print(f"Финальный размер: {df.shape}")
print(f"Колонки: {df.columns.tolist()}")

До фильтрации: (22676, 12)
После удаления дубликатов: (20841, 12)
После фильтра 'Secondary': (12227, 12)
После фильтра 'Moscow': (11593, 12)
Финальный размер: (11593, 10)
Колонки: ['Price', 'Metro station', 'Minutes to metro', 'Number of rooms', 'Area', 'Living area', 'Kitchen area', 'Floor', 'Number of floors', 'Renovation']


### Изучение колонки Metro station: слишком много уникальных значений

In [47]:
# Сколько уникальных станций осталось после фильтрации Москвы
print(f"Уникальных станций после фильтра Москва: {df['Metro station'].nunique()}")

# Топ-30 станций по количеству объявлений
print("\nТоп-30 станций:")
print(df['Metro station'].value_counts().head(30))

# Хвост — станции с минимальным числом объявлений
print("\nСтанции с 1-2 объявлениями (вероятные опечатки/электрички):")
station_counts = df['Metro station'].value_counts()
rare_stations = station_counts[station_counts <= 2]
print(f"Таких станций: {len(rare_stations)}")
print(rare_stations.head(30))

# Проверка на пробелы и регистр
print("\nПримеры названий с возможными опечатками:")
stations_list = df['Metro station'].unique()
# Ищем пары станций, отличающихся только пробелами/регистром
normalized = {}
for s in stations_list:
    key = s.strip().lower()
    normalized.setdefault(key, []).append(s)
duplicates = {k: v for k, v in normalized.items() if len(v) > 1}
print(f"Пар с возможными опечатками: {len(duplicates)}")
for k, v in list(duplicates.items())[:10]:
    print(f"  {v}")


Уникальных станций после фильтра Москва: 542

Топ-30 станций:
Metro station
ЗИЛ                       205
Аминьевская               150
Некрасовка                130
Панфиловская              121
Народное Ополчение        120
Минская                   118
Отрадное                  117
Селигерская               111
Коммунарка                107
Смоленская                106
Щелковская                100
Прокшино                   86
Давыдково                  84
Шелепиха                   84
 Спортивная                82
Люблино                    82
Фили                       78
Улица 1905 года            77
Новаторская                76
Динамо                     75
 Улица 1905 года           74
Арбатская                  70
Беломорская                70
Раменки                    69
Аэропорт                   69
Новогиреево                68
 Минская                   67
Бунинская Аллея            67
Преображенская площадь     66
Академическая              66
Name: count, dtype: int6

# Исправление столбца Metro station

In [48]:
# Сохраняем число уникальных значений ДО для статистики
n_before = df['Metro station'].nunique()
print(f"Уникальных станций до очистки: {n_before}")

# Нормализация: убираем пробелы по краям и внутренние двойные пробелы
df['Metro station'] = (
    df['Metro station']
    # пробелы по краям
    .str.strip()
    # двойные пробелы внутри
    .str.replace(r'\s+', ' ', regex=True)
)

n_after_strip = df['Metro station'].nunique()
print(f"После strip и нормализации пробелов: {n_after_strip}")
print(f"Склеено: {n_before - n_after_strip} дубликатов")


Уникальных станций до очистки: 542
После strip и нормализации пробелов: 313
Склеено: 229 дубликатов


In [49]:
# Ищем пары после strip, различающиеся только регистром
stations = df['Metro station'].unique()
case_groups = {}
for s in stations:
    key = s.lower()
    case_groups.setdefault(key, []).append(s)
case_duplicates = {k: v for k, v in case_groups.items() if len(v) > 1}

print(f"Пар, различающихся только регистром: {len(case_duplicates)}")
for k, v in list(case_duplicates.items())[:20]:
    print(f"  {v}")


Пар, различающихся только регистром: 7
  ['Парк Культуры', 'Парк культуры']
  ['Петровский Парк', 'Петровский парк']
  ['Рабочий посёлок', 'Рабочий Посёлок']
  ['Бунинская аллея', 'Бунинская Аллея']
  ['Шоссе Энтузиастов', 'Шоссе энтузиастов']
  ['Соколиная гора', 'Соколиная Гора']
  ['Верхние котлы', 'Верхние Котлы']


In [50]:
# Для каждой группы регистровых дубликатов берём самый частотный вариант написания
station_counts = df['Metro station'].value_counts()

replace_map = {}
for key, variants in case_duplicates.items():
    # Самый частотный вариант становится каноническим
    canonical = max(variants, key=lambda v: station_counts[v])
    for v in variants:
        if v != canonical:
            replace_map[v] = canonical

df['Metro station'] = df['Metro station'].replace(replace_map)
yo_after_case = df['Metro station'].nunique()
print(f"После нормализации регистра: {yo_after_case}")
print(f"Склеено ещё: {n_after_strip - yo_after_case}")


После нормализации регистра: 306
Склеено ещё: 7


In [51]:
# Проверка на букву "ё"
stations = df['Metro station'].unique()
yo_groups = {}
for s in stations:
    key = s.lower().replace('ё', 'е')
    yo_groups.setdefault(key, []).append(s)
yo_duplicates = {k: v for k, v in yo_groups.items() if len(v) > 1}

print(f"Пар, различающихся буквой ё/е: {len(yo_duplicates)}")
for v in yo_duplicates.values():
    print(f"  {v}")


Пар, различающихся буквой ё/е: 10
  ['Молодёжная', 'Молодежная']
  ['Филёвский парк', 'Филевский парк']
  ['Хорошёво', 'Хорошево']
  ['Новые Черёмушки', 'Новые Черемушки']
  ['Щёлковская', 'Щелковская']
  ['Савёловская', 'Савеловская']
  ['Тропарёво', 'Тропарево']
  ['Семёновская', 'Семеновская']
  ['Тёплый Стан', 'Теплый Стан']
  ['Воробьёвы горы', 'Воробьевы горы']


In [52]:
# Для каждой группы написания "ё" берём самый частотный вариант написания
station_counts = df['Metro station'].value_counts()

replace_map = {}
for key, variants in yo_duplicates.items():
    # Самый частотный вариант становится каноническим
    canonical = max(variants, key=lambda v: station_counts[v])
    for v in variants:
        if v != canonical:
            replace_map[v] = canonical

df['Metro station'] = df['Metro station'].replace(replace_map)
yo_after_case = df['Metro station'].nunique()
print(f"После переработки \"е/ё\": {yo_after_case}")
print(f"Склеено ещё: {n_after_strip - yo_after_case}")

После переработки "е/ё": 296
Склеено ещё: 17


In [53]:
# После нормализаций смотрим на оставшиеся редкие станции
station_counts = df['Metro station'].value_counts()
print(f"Итого уникальных станций: {len(station_counts)}")
print(f"Станций с 1-2 объявлениями: {(station_counts <= 2).sum()}")
print(f"Станций с 1 объявлением: {(station_counts == 1).sum()}")
print("\nСтанции с 1 объявлением:")
print(station_counts[station_counts == 1].head(50))


Итого уникальных станций: 296
Станций с 1-2 объявлениями: 17
Станций с 1 объявлением: 13

Станции с 1 объявлением:
Metro station
Выставочный центр           1
Подольск                    1
Санино                      1
Вешняки                     1
Улица Академика Королёва    1
Хлебниково                  1
Партизанская                1
Долгопрудная                1
Лубянка                     1
Терехово (Мнёвники)         1
Мякинино                    1
Сколково                    1
Битца                       1
Name: count, dtype: int64


## Проверка логики

In [54]:
# 1. Площадь: living + kitchen не должны превышать общую
illogical_area = df[df['Living area'] + df['Kitchen area'] > df['Area']]
print(f"Строк, где Living + Kitchen > Area: {len(illogical_area)}")
if len(illogical_area) > 0:
    print(illogical_area[['Area', 'Living area', 'Kitchen area']].head(10))

# 2. Этаж не должен превышать этажность
illogical_floor = df[df['Floor'] > df['Number of floors']]
print(f"\nСтрок, где Floor > Number of floors: {len(illogical_floor)}")
if len(illogical_floor) > 0:
    print(illogical_floor[['Floor', 'Number of floors']].head(10))

# 3. Распределение по числу комнат (включая студии)
print(f"\nРаспределение по числу комнат:")
print(df['Number of rooms'].value_counts().sort_index())

# 4. Сколько объектов с нулевыми Living area или Kitchen area
print(f"\nLiving area = 0: {(df['Living area'] == 0).sum()}")
print(f"Kitchen area = 0: {(df['Kitchen area'] == 0).sum()}")
print(f"Minutes to metro = 0: {(df['Minutes to metro'] == 0).sum()}")


Строк, где Living + Kitchen > Area: 855
      Area  Living area  Kitchen area
489  178.0        165.0          21.4
613  379.5        370.0          38.0
692  157.0        157.0          19.6
711   62.0         62.0          11.8
773   32.0         29.5           9.3
791   21.0         12.8           8.4
856   14.2          9.3           7.8
857   15.5         13.0           7.9
858   17.3         14.0           8.1
860   15.1          9.8           7.9

Строк, где Floor > Number of floors: 131
      Floor  Number of floors
2852    5.0                 4
2916   16.0                 5
2958   18.0                 5
3038   18.0                17
3135   18.0                17
3278   10.0                 6
3299    9.0                 6
3499    8.0                 5
3521   18.0                16
3585    6.0                 5

Распределение по числу комнат:
Number of rooms
0.0     2113
1.0     2050
2.0     2136
3.0     2240
4.0     1640
5.0      845
6.0      523
7.0       27
8.0       10
9.0  

## Исправление Living area и Kitchen area: исключаем столбцы из датасета

In [55]:
# Фиксируем число проблемных строк для отчёта, до удаления колонок
n_illogical_area = ((df['Living area'] + df['Kitchen area']) > df['Area']).sum()
print(f"Логически некорректных строк (Living+Kitchen > Area): {n_illogical_area}")

# Удаляем колонки Living area и Kitchen area
df = df.drop(columns=['Living area', 'Kitchen area'])
print(f"Колонки после удаления: {df.columns.tolist()}")
print(f"Размер: {df.shape}")


Логически некорректных строк (Living+Kitchen > Area): 855
Колонки после удаления: ['Price', 'Metro station', 'Minutes to metro', 'Number of rooms', 'Area', 'Floor', 'Number of floors', 'Renovation']
Размер: (11593, 8)


## Удаляем все проблемные строки Floor и Number of floors

In [56]:
n_illogical_floor = (df['Floor'] > df['Number of floors']).sum()
print(f"Строк с Floor > Number of floors: {n_illogical_floor}")

df = df[df['Floor'] <= df['Number of floors']].reset_index(drop=True)
print(f"Размер после удаления: {df.shape}")


Строк с Floor > Number of floors: 131
Размер после удаления: (11462, 8)


## Исправление Minutes to metro = 0: заменяем на медиану по станции

In [57]:
# Заменяем 0 на NaN, потом на медиану по станции
df.loc[df['Minutes to metro'] == 0, 'Minutes to metro'] = np.nan
df['Minutes to metro'] = df.groupby('Metro station')['Minutes to metro'].transform(
    lambda x: x.fillna(x.median())
)
# Если станция вся была с нулями — заполнить общей медианой
df['Minutes to metro'] = df['Minutes to metro'].fillna(df['Minutes to metro'].median())
print(f"После замены: minutes=0 → {(df['Minutes to metro'] == 0).sum()}")


После замены: minutes=0 → 0


## Итоговая версия после переработки данных

In [58]:
print(f"Размер данных: {df.shape}")
df.describe()

Размер данных: (11462, 8)


,Price,Minutes to metro,Number of rooms,Area,Floor,Number of floors
count,1.146200e+04,11462.000000,11462.000000,11462.000000,11462.000000,11462.000000
mean,5.435456e+07,12.406212,2.352992,85.276437,8.694818,17.332577
std,1.022255e+08,7.155302,1.745769,81.183651,8.483289,12.230825
min,1.150000e+06,1.000000,0.000000,6.000000,1.000000,1.000000
25%,1.199000e+07,7.000000,1.000000,38.000000,3.000000,9.000000
50%,2.000000e+07,11.000000,2.000000,60.000000,6.000000,15.000000
75%,4.999000e+07,16.000000,4.000000,104.800000,12.000000,22.000000
max,2.455020e+09,60.000000,12.000000,1117.000000,92.000000,97.000000


## Сохранение после первичной обработки

In [59]:
df.to_csv('dataset/cleaned_dataset.csv', index=False)